# 03 SchNet on QM9



In this tutorial we are going to install:

1. Install and import **SchNetPack** and dependencies.
2. Download a small subset of the **QM9** molecule dataset.
3. Build a **SchNet** neural network potential.
4. Train it to predict the **U0 (internal energy at 0 K)**.
5. Use the trained model to make predictions on:
   - QM9 test molecules
   - a custom methane-like molecule defined with **ASE**.



## 3.1 Environment & Installation

In this step, we install a clean and compatible set of machine-learning libraries to prepare our Colab environment for training SchNet.  
We first install specific versions of **PyTorch**, **torchvision**, and **torchaudio** to avoid conflicts with the versions pre-installed on Colab.  
Then we install **SchNetPack** together with **PyTorch Lightning**, **TorchMetrics**, and **ASE**, which provide the tools needed for molecular simulations, training neural networks, and handling atomic structures.

We use **PyTorch Lightning** to simplify the training process. Lightning handles the repetitive boilerplate involved in training neural etworks—such as writing training/validation loops, tracking metrics, saving checkpoints, and managing CPU/GPU devices. This allows us to focus on the **model architecture and data**, while Lightning takes care of the training mechanics behind the scenes.  


In [ ]:
!pip install -q torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1


!pip install -q schnetpack pytorch-lightning torchmetrics ase


## 3.2 Imports and basic setup

Here we import:
- **schnetpack** (core library)
- **QM9** dataset helper
- **transforms** (neighbor lists, normalization, etc.)
- **PyTorch** and **PyTorch Lightning** (for training)
- Some basic utilities (paths, device, etc.)


In [ ]:
import os

import schnetpack as spk
from schnetpack.datasets import QM9
import schnetpack.transform as trn

import torch
import torchmetrics
import pytorch_lightning as pl

# Directory where we store data, logs, and trained models
workdir = "./qm9_schnet_tutorial"
os.makedirs(workdir, exist_ok=True)

# Select device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 3.3 Load a small QM9 dataset



SchNetPack ships with convenience dataset classes.  
Here we use the **QM9** benchmark dataset:

- `batch_size=100` — mini-batch size for training.
- `num_train`, `num_val` — we use a small subset to keep the tutorial fast.
- `transforms`:
  - `ASENeighborList` builds neighbor lists (who is near whom).
  - `RemoveOffsets` subtracts per-atom reference energies + mean (helps learning).
  - `CastTo32` casts to `float32` for speed on GPUs.
- We **only** load `QM9.U0` (internal energy at 0 K).

After defining the dataset, we call:
- `prepare_data()` → download/convert if needed.
- `setup()` → split into train/val/test sets.


In [ ]:
# Remove any old split file so that the tutorial is reproducible
split_file = os.path.join(workdir, "split.npz")
if os.path.exists(split_file):
    os.remove(split_file)

qm9data = QM9(
    datapath="./qm9_data",      # folder where QM9 will be downloaded & stored
    batch_size=100,             # batch size
    num_train=1000,             # use 1000 molecules for training (small!)
    num_val=1000,               # 1000 for validation
    # num_test will be "rest" by default
    transforms=[
        trn.ASENeighborList(cutoff=5.0),
        trn.RemoveOffsets(QM9.U0, remove_mean=True, remove_atomrefs=True),
        trn.CastTo32(),
    ],
    property_units={QM9.U0: "eV"},   # make sure energies are in eV
    num_workers=1,                   # increase for more CPU cores
    split_file=split_file,           # where the split is stored
    pin_memory=True,                 # set False if no GPU
    load_properties=[QM9.U0],        # only load U0
)

qm9data.prepare_data()
qm9data.setup()

print("Total entries:", len(qm9data.dataset))
print("Train:", len(qm9data.train_dataset))
print("Validation:", len(qm9data.val_dataset))
print("Test:", len(qm9data.test_dataset))
print("Available properties:", qm9data.dataset.available_properties)


After downloading QM9, SchNetPack reports how the dataset is split and what properties are available.

- **Total entries: 133,885**  
  This is the full QM9 dataset: ~134k molecules with DFT-calculated properties.

- **Train: 1000**  
  We selected 1,000 molecules for training.  
  This small subset keeps the tutorial fast.

- **Validation: 1000**  
  Another 1,000 molecules used only for checking the model’s performance during training. They are never used to update the network weights.

- **Test: 131,885**  
  All remaining molecules. These are used to evaluate the final model.

We use a small training subset and a very large test set in QM9 because scientific datasets are big and expensive to train on, so we train quickly on fewer examples and evaluate reliably on the rest.

- **Available properties**  
  QM9 provides many quantum chemistry properties, including:  
  - `energy_U0`: internal energy at 0 K  
  - `energy_U`: internal energy at 298 K  
  - `enthalpy_H`, `free_energy`, `heat_capacity`  
  - `homo`, `lumo`, `gap`  
  - `dipole_moment`, `isotropic_polarizability`  
  - rotational constants and more

In this tutorial, we focus on **`energy_U0`**, because it is one of the simplest and most stable properties to learn.


## 3.4 Inspect atomic reference values & statistics

 SchNetPack uses **atomic reference energies** to normalize molecular energies. Instead of predicting the full (very large) molecular energy, it models only a small correction:

$\
E_{\text{molecule}} \approx \sum_i E_{\text{atomref}}(Z_i) + \Delta E
$

- The **sum of atomic reference energies** provides the large baseline.  
- The neural network learns only the **small correction** \($\Delta E$\), which makes training easier and more stable.

So each atom type (H, C, O, etc.) has a characteristic baseline energy, and molecular energies can be expressed as a sum of these atomic contributions plus a small correction that the neural network learns.

 We print the reference energies for hydrogen, carbon, and oxygen, and then compute the mean and standard deviation of the atomization energy per atom, which helps us understand the scale and natural variability of the target property.


In [ ]:
# Per-element reference energies (in eV)
atomrefs = qm9data.train_dataset.atomrefs
print(f"U0 of H (Z=1): {atomrefs[QM9.U0][1].item():.2f} eV")
print(f"U0 of C (Z=6): {atomrefs[QM9.U0][6].item():.2f} eV")
print(f"U0 of O (Z=8): {atomrefs[QM9.U0][8].item():.2f} eV")

# Mean and std deviation of atomization energy per atom
means, stddevs = qm9data.get_stats(
    QM9.U0, divide_by_atoms=True, remove_atomref=True
)
print(f"Mean atomization energy / atom: {means.item():.2f}")
print(f"Std. dev. atomization energy / atom: {stddevs.item():.2f}")


- **Atomic reference energies** (H, C, O) are large negative baselines that get subtracted so the model learns only small energy corrections.  
- **Mean atomization energy per atom (–4.25 eV)** shows the average corrected molecular stability.  
- **Std. dev. (0.19 eV)** indicates that these corrected energies vary only a little, making the learning task easier.

## 3.5 Build the SchNet model

**SchNet Model Configuration**

A SchNet potential is built from three main components:

1. **Input modules**  
   These compute geometric information needed by the model, such as pairwise distances between atoms.

2. **Representation**  
   The core of the model (`spk.representation.SchNet`), which uses:  
   - 3 interaction (message-passing) layers  
   - a cosine cutoff at 5 Å  
   - a Gaussian radial basis with 20 functions  
   - 30-dimensional atom-wise feature vectors  

3. **Output module**  
   An `Atomwise` head that predicts atomic contributions, which are then summed to obtain the molecular energy **U0**.

Additionally, we use **postprocessors**, which run only during *inference mode* (`model.inference_mode = True`):  
- `CastTo64` converts outputs to float64. Molecular energies are small corrections added to very large atomic reference energies, and using 64-bit precision avoids rounding errors that can occur with float32.
- `AddOffsets` adds back the mean and atomic reference energies removed during preprocessing  

These steps ensure numerically stable and physically meaningful predictions.

---

**Parameter Table**

The table below summarizes the exact configuration used in this tutorial:

| Component | Parameter | Value | Meaning |
|----------|-----------|--------|---------|
| **Representation (SchNet)** | `n_atom_basis` | **30** | Size of atomic feature vectors (embedding dimension). |
| | `n_interactions` | **3** | Number of interaction (message-passing) blocks. |
| | `radial_basis` | GaussianRBF (20 functions) | Expands interatomic distances into 20 radial basis functions. |
| | `cutoff_fn` | CosineCutoff(5.0 Å) | Smoothly cuts off interactions beyond 5 Å. |
| **Radial Basis** | `n_rbf` | **20** | Number of Gaussian basis functions. |
| | `cutoff` | **5.0 Å** | Maximum interaction distance. |
| **Input Module** | PairwiseDistances | — | Computes all interatomic distances within the cutoff. |
| **Output Module** | Atomwise | — | Predicts atom-wise energy contributions. |
| | `n_in` | **30** | Input feature dimension (matches `n_atom_basis`). |
| | `output_key` | `QM9.U0` | Predicted property: internal energy at 0 K. |
| **Postprocessors** | CastTo64 | — | Converts predicted energy to float64. |
| | AddOffsets | add_mean=True, add_atomrefs=True | Restores reference energies and mean removed earlier. |
| **Overall Model** | NeuralNetworkPotential | — | Combines representation, input, output, and postprocessing. |

---

- SchNet uses 3 message-passing blocks to learn atomic interactions.  
- A 5 Å cutoff and 20 Gaussian radial basis functions describe local environments.  
- The model predicts **U0** (internal energy at 0 K) via atom-wise energy contributions.  
- During inference, reference energies and the dataset mean are added back to recover the full physical energy.  


In [ ]:
cutoff = 5.0
n_atom_basis = 30

# 1) Input module: compute pairwise distances
pairwise_distance = spk.atomistic.PairwiseDistances()

# 2) Radial basis and SchNet representation
radial_basis = spk.nn.GaussianRBF(n_rbf=20, cutoff=cutoff)

schnet = spk.representation.SchNet(
    n_atom_basis=n_atom_basis,
    n_interactions=3,  # number of interaction blocks
    radial_basis=radial_basis,
    cutoff_fn=spk.nn.CosineCutoff(cutoff),
)

# 3) Atomwise energy prediction module
pred_U0 = spk.atomistic.Atomwise(
    n_in=n_atom_basis,
    output_key=QM9.U0,  # this key will be used in the output dict
)

# Combine everything into a NeuralNetworkPotential
nnpot = spk.model.NeuralNetworkPotential(
    representation=schnet, # 1
    input_modules=[pairwise_distance], # 2
    output_modules=[pred_U0], #3
    postprocessors=[
        trn.CastTo64(),  # cast outputs to float64
        trn.AddOffsets(QM9.U0, add_mean=True, add_atomrefs=True),
    ],
)

# Move model to the right device
nnpot.to(device)
print(nnpot)


## 3.6 Define loss, metrics, and training task

In this step, we specify what the model should learn and how it should be optimized. We create a `ModelOutput` that tells SchNetPack to predict the **U0** energy and to train using **mean squared error (MSE)**, while monitoring **mean absolute error (MAE)** as an evaluation metric. We then wrap everything into an `AtomisticTask`, which sets up the training loop and optimizer (AdamW with a learning rate of 1e-4). This task object is what PyTorch Lightning uses to run the full training process.


In [ ]:
# Describe the output: which loss & metric to use
output_U0 = spk.task.ModelOutput(
    name=QM9.U0,                         # target key in the batch dict
    loss_fn=torch.nn.MSELoss(),          # regression loss
    loss_weight=1.0,                     # weight if multiple outputs
    metrics={"MAE": torchmetrics.MeanAbsoluteError()},
)

# Wrap into an AtomisticTask (LightningModule)
task = spk.task.AtomisticTask(
    model=nnpot,
    outputs=[output_U0],
    optimizer_cls=torch.optim.AdamW,
    optimizer_args={"lr": 1e-4},
)


## 3.7 Training the SchNet Model

In this step, we prepare and run the full training loop using PyTorch Lightning. Here is what each part of the code does:

1. **Create a TensorBoard logger**  
   Records training and validation metrics so you can visualize the learning curves.

2. **Create a ModelCheckpoint callback**  
   Automatically saves the best model based on the validation loss during training.

3. **Initialize the Lightning Trainer**  
   Controls the training loop, selects GPU/CPU, applies callbacks, and sets the number of epochs.

4. **Run the training**  
   `trainer.fit(task, datamodule=qm9data)` starts the actual optimization using our model and the QM9 dataset.

For this tutorial, we train for only **3 epochs**, but you can increase this number later to achieve better accuracy.


In [ ]:
logger = pl.loggers.TensorBoardLogger(save_dir=workdir)

callbacks = [
    spk.train.ModelCheckpoint(
        model_path=os.path.join(workdir, "best_inference_model"),
        save_top_k=1,
        monitor="val_loss",
    )
]

trainer = pl.Trainer(
    callbacks=callbacks,
    logger=logger,
    default_root_dir=workdir,
    max_epochs=3,      # increase for better accuracy
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
)

trainer.fit(task, datamodule=qm9data)


* The training ran successfully for 3 epochs using CPU.  
* The model contains **16.4k trainable parameters**, making it lightweight and fast to train.  
* Lightning reported that validation metrics were logged only once per epoch because our small dataset produces just 10 training batches.  
* After completing all epochs (`max_epochs=3`), the best checkpoint was saved based on the lowest validation loss, which reached approximately **37.7**.

## 3.8 Evaluating the Trained Model

* The checkpoint callback saved an **inference-ready SchNet model** that automatically restores mean and atomic reference energies.  
Here, we:
* (1) load the best inference model,
* (2) take one batch from the test set, and
* (3) compare its predicted **U0** energies with the true reference values.  

By printing the predictions for a few molecules, we can quickly check how well the trained model generalizes to unseen data.

In [ ]:
from schnetpack.utils.compatibility import load_model

# Load best inference model (CPU is fine for demo)
best_model = load_model(
    os.path.join(workdir, "best_inference_model"),
    device=device,
)

# Put the model into inference mode
best_model.inference_mode = True
best_model.to(device)

# Take one batch from the test set and compare
test_batch = next(iter(qm9data.test_dataloader()))
for key in test_batch:
    test_batch[key] = test_batch[key].to(device)

with torch.no_grad():
    pred = best_model(test_batch)

u0_pred = pred[QM9.U0].cpu().numpy()
u0_ref = test_batch[QM9.U0].cpu().numpy()

for i in range(5):  # show a few examples
    print(f"Molecule {i}: U0_pred = {u0_pred[i]: .2f} eV,  U0_ref = {u0_ref[i]: .2f} eV")


In [ ]:
# Move predictions and reference energies to CPU and convert to NumPy for plotting
u0_pred = pred[QM9.U0].cpu().numpy()
u0_ref = test_batch[QM9.U0].cpu().numpy()

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from math import sqrt

best_model.eval()
best_model.inference_mode = True

N = 10000   # evaluate only 10k test molecules
count = 0

all_ref = []
all_pred = []

with torch.no_grad():
    for batch in qm9data.test_dataloader():
        # stop early
        if count >= N:
            break

        for key in batch:
            batch[key] = batch[key].to(device)

        out = best_model(batch)

        all_pred.append(out[QM9.U0].cpu().numpy())
        all_ref.append(batch[QM9.U0].cpu().numpy())

        count += len(batch[QM9.U0])

# Concatenate
all_ref = np.concatenate(all_ref)
all_pred = np.concatenate(all_pred)

# Metrics
mae = np.mean(np.abs(all_pred - all_ref))
rmse = sqrt(np.mean((all_pred - all_ref)**2))

print(f"Evaluated {len(all_ref)} molecules.")
print(f"MAE:  {mae:.3f} eV")
print(f"RMSE: {rmse:.3f} eV")

# Parity plot (subsampled)
n_plot = min(2000, len(all_ref))
idx = np.random.choice(len(all_ref), size=n_plot, replace=False)

plt.figure(figsize=(5, 5))
plt.scatter(all_ref[idx], all_pred[idx], alpha=0.4)
plt.plot([all_ref.min(), all_ref.max()],
         [all_ref.min(), all_ref.max()],
         linestyle="--")
plt.xlabel("Reference U0 (eV)")
plt.ylabel("Predicted U0 (eV)")
plt.title("SchNet Parity Plot (10k test sample)")
plt.grid(True)
plt.tight_layout()
plt.show()


## 3.9 Predict energy for methane molecule



Often we want to evaluate a trained potential on **new structures**.
We’ll:

1. Create a **CH₄-like structure** using **ASE**.
2. Convert it to SchNetPack inputs using `AtomsConverter`.
3. Run the `best_model` and get a predicted energy.
4. Optionally, use `SpkCalculator` to integrate with ASE’s `atoms.get_total_energy()`.


In [ ]:

import numpy as np
from ase import Atoms

# Converter: takes ASE Atoms and builds the batch dict SchNetPack expects
converter = spk.interfaces.AtomsConverter(
    neighbor_list=trn.ASENeighborList(cutoff=5.0),
    dtype=torch.float32,
    device=device,
)

# Simple methane: one carbon at origin, four hydrogens arranged around it
numbers = np.array([6, 1, 1, 1, 1])  # atomic numbers: C, H, H, H, H
positions = np.array(
    [
        [0.0000, 0.0000, 0.0000],   # C
        [0.6291, 0.6291, 0.6291],   # H
        [-0.6291, -0.6291, 0.6291], # H
        [-0.6291, 0.6291, -0.6291], # H
        [0.6291, -0.6291, -0.6291], # H
    ]
)

atoms = Atoms(numbers=numbers, positions=positions)

# Convert to SchNetPack input format and predict
inputs = converter(atoms)

with torch.no_grad():
    custom_pred = best_model(inputs)

print("Predicted U0 for methane-like molecule:", custom_pred[QM9.U0].item(), "eV")


## 3.10 Inspecting a Random QM9 Test Molecule


#
In this step, we pick a **random molecule from the QM9 test set** and compare the
model’s prediction with the corresponding target value used during training.
Instead of manually reconstructing energies, we take a batch directly from the
test dataloader so that **all preprocessing and normalization steps** match the
model’s expectations. We then extract the original atomic structure, compute its
predicted U0 energy with SchNet, and compare it to the reference energy provided
in the batch. The selected structure is also saved as an
`.xyz` file for visualization.


In [ ]:
import random
from ase import Atoms

best_model.eval()
best_model.inference_mode = True  # use the same mode as in test evaluation

# Take one batch from the test loader
batch = next(iter(qm9data.test_dataloader()))

# Move batch to device
for key in batch:
    batch[key] = batch[key].to(device)

with torch.no_grad():
    pred = best_model(batch)[QM9.U0]   # predicted energies
    ref = batch[QM9.U0]                # reference energies on the same scale

# Pick a random index within this batch
i = random.randint(0, len(ref) - 1)

# Get original dataset index to reconstruct atoms and formula
original_idx = batch["_idx"][i].item()
entry = qm9data.dataset[original_idx]

Z = entry["_atomic_numbers"].cpu().numpy()
pos = entry["_positions"].cpu().numpy()
atoms_qm9 = Atoms(numbers=Z, positions=pos)

true_u0 = ref[i].item()
pred_u0 = pred[i].item()

print(f"Random QM9 test example (original index): {original_idx}")
print("Chemical formula:", atoms_qm9.get_chemical_formula())
print(f"True U0 (QM9, model scale):      {true_u0:.3f} eV")
print(f"Predicted U0 (SchNet):           {pred_u0:.3f} eV")
print(f"Error (SchNet - QM9):            {pred_u0 - true_u0:.3f} eV")

atoms_qm9.write("qm9_example.xyz")
print('Saved structure to "qm9_example.xyz".')


In [ ]:
plt.figure(figsize=(4,4))
plot_atoms(atoms_qm9, rotation=('90x, 0y, 0z'))  # rotate molecule onto XY plane
plt.axis('off')
plt.show()
